### Regression Analysis & Demand Drivers
**E-Commerce Demand Forecasting & Supply Chain Optimization**

Goals:
- Identify key demand drivers (price, seasonality, promotions)
- Quantify price elasticity and promotional lift
- Validate with ANOVA and residual diagnostics
- Isolate statistically significant predictors

In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Regression & Stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.gofplots import qqplot
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid')
print('Libraries loaded')

In [ ]:
# Cell 2 — Load Data
demand = pd.read_csv('../data/processed/demand_clean.csv', parse_dates=['date'])
retail = pd.read_csv('../data/processed/retail_clean.csv', parse_dates=['date'])

print('Demand shape:', demand.shape)
print('Retail shape:', retail.shape)
demand.head(3)

In [ ]:
# Cell 3 — Feature Engineering for Regression
df = demand.copy()

# Time features
df['month']      = df['date'].dt.month
df['dow']        = df['date'].dt.dayofweek
df['quarter']    = df['date'].dt.quarter
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['year']       = df['date'].dt.year
df['dayofyear']  = df['date'].dt.dayofyear

# Is weekend
df['is_weekend'] = (df['dow'] >= 5).astype(int)

# Is holiday season (Nov-Dec)
df['is_holiday_season'] = df['month'].isin([11, 12]).astype(int)

# Is summer peak (Jun-Aug)
df['is_summer'] = df['month'].isin([6, 7, 8]).astype(int)

# Lag features
daily = df.groupby('date').agg({'sales':'sum'}).reset_index()
daily['lag_7']   = daily['sales'].shift(7)
daily['lag_14']  = daily['sales'].shift(14)
daily['roll_7']  = daily['sales'].shift(1).rolling(7).mean()
daily['roll_30'] = daily['sales'].shift(1).rolling(30).mean()

# Merge lag features back
df = df.merge(daily[['date','lag_7','lag_14','roll_7','roll_30']], on='date', how='left')
df = df.dropna()

print('Features engineered')
print('Shape after feature engineering:', df.shape)
df.head(3)

In [ ]:
# Cell 4 — Price Data from Retail (Elasticity Features)
# Aggregate price and quantity from retail dataset
daily_retail = retail.groupby('date').agg(
    avg_price  = ('Price', 'mean'),
    total_qty  = ('Quantity', 'sum'),
    revenue    = ('Revenue', 'sum')
).reset_index()
daily_retail['date'] = pd.to_datetime(daily_retail['date'])

print('Daily retail aggregated')
print(daily_retail.shape)
daily_retail.head(3)

In [ ]:
# Cell 5 — OLS Regression: Demand Drivers (Demand Dataset)
reg_features = ['month', 'dow', 'quarter', 'is_weekend',
                'is_holiday_season', 'is_summer',
                'lag_7', 'lag_14', 'roll_7', 'roll_30']

X = sm.add_constant(df[reg_features])
y = df['sales']

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

In [ ]:
# Cell 6 — Significant Predictors (p < 0.05)
pvals = ols_model.pvalues.drop('const')
sig   = pvals[pvals < 0.05].sort_values()
insig = pvals[pvals >= 0.05].sort_values()

print(f' Significant predictors ({len(sig)}):')
for feat, p in sig.items():
    coef = ols_model.params[feat]
    print(f'   {feat:<22} coef={coef:+.3f}  p={p:.4f}')

print(f'\n Not significant ({len(insig)}):')
for feat, p in insig.items():
    print(f'   {feat:<22} p={p:.4f}')

print(f'\nR-squared  : {ols_model.rsquared:.4f}')
print(f'Adj R²     : {ols_model.rsquared_adj:.4f}')
print(f'F-statistic: {ols_model.fvalue:.2f}  (p={ols_model.f_pvalue:.4e})')

In [ ]:
# Cell 7 — Coefficient Plot
coef_df = pd.DataFrame({
    'feature': ols_model.params.index[1:],
    'coef':    ols_model.params.values[1:],
    'pval':    ols_model.pvalues.values[1:]
})
coef_df['significant'] = coef_df['pval'] < 0.05
coef_df['color']       = coef_df['significant'].map({True: '#1a936f', False: '#ccc'})
coef_df = coef_df.sort_values('coef')

fig = px.bar(coef_df, x='coef', y='feature', orientation='h',
             color='significant',
             color_discrete_map={True: '#1a936f', False: '#e0e0e0'},
             title='OLS Regression Coefficients (Green = Significant at p<0.05)',
             labels={'coef': 'Coefficient', 'feature': 'Feature'})
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(showgrid=False), yaxis_title='')
fig.show()

In [ ]:
# Cell 8 — ANOVA: Sales by Month and Day of Week
print('=== ANOVA: Sales ~ Month ===')
groups_month = [df[df['month'] == m]['sales'].values for m in range(1, 13)]
f_month, p_month = stats.f_oneway(*groups_month)
print(f'F-statistic : {f_month:.4f}')
print(f'P-value     : {p_month:.4e}')
print('→ Month is', 'a SIGNIFICANT' if p_month < 0.05 else 'NOT a significant', 'demand driver')

print()
print('=== ANOVA: Sales ~ Day of Week ===')
groups_dow = [df[df['dow'] == d]['sales'].values for d in range(7)]
f_dow, p_dow = stats.f_oneway(*groups_dow)
print(f'F-statistic : {f_dow:.4f}')
print(f'P-value     : {p_dow:.4e}')
print('→ Day of week is', 'a SIGNIFICANT' if p_dow < 0.05 else 'NOT a significant', 'demand driver')

print()
print('=== ANOVA: Sales ~ Store ===')
groups_store = [df[df['store'] == s]['sales'].values for s in df['store'].unique()]
f_store, p_store = stats.f_oneway(*groups_store)
print(f'F-statistic : {f_store:.4f}')
print(f'P-value     : {p_store:.4e}')
print('→ Store is', 'a SIGNIFICANT' if p_store < 0.05 else 'NOT a significant', 'demand driver')

In [ ]:
# Cell 9 — Sales Distribution by Month (Boxplot)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['month_name'] = df['month'].apply(lambda x: month_names[x-1])

fig = px.box(df, x='month_name', y='sales',
             title='Sales Distribution by Month — ANOVA Visualization',
             labels={'sales': 'Sales', 'month_name': 'Month'},
             color='month_name',
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_layout(height=430, plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False, xaxis=dict(showgrid=False))
fig.show()

In [ ]:
# Cell 10 — Price Elasticity (Online Retail)
price_qty = daily_retail[['avg_price','total_qty']].copy()
price_qty = price_qty[price_qty['avg_price'] < price_qty['avg_price'].quantile(0.95)]

log_price = np.log(price_qty['avg_price'] + 1)
log_qty   = np.log(price_qty['total_qty'] + 1)

slope, intercept, r, p, se = stats.linregress(log_price, log_qty)

print('=== Price Elasticity of Demand ===')
print(f'Elasticity Coefficient : {slope:.4f}')
print(f'R-squared              : {r**2:.4f}')
print(f'P-value                : {p:.4f}')
print(f'Std Error              : {se:.4f}')
print()
if slope < -1:
    print('→ ELASTIC: 1% price increase → more than 1% demand drop')
elif -1 <= slope < 0:
    print('→ INELASTIC: price changes have modest effect on demand')
else:
    print('→ Positive or near-zero elasticity detected')

fig = px.scatter(price_qty, x='avg_price', y='total_qty', trendline='ols',
                 title=f'Price vs Quantity — Elasticity Coefficient: {slope:.3f}',
                 labels={'avg_price': 'Avg Price ($)', 'total_qty': 'Total Qty Sold'},
                 color_discrete_sequence=['#e74c3c'])
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Cell 11 — Ridge & Lasso Regression (Regularization)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(df[reg_features])
y_vals   = df['sales'].values

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

ridge_cv = cross_val_score(ridge, X_scaled, y_vals, cv=5, scoring='r2')
lasso_cv = cross_val_score(lasso, X_scaled, y_vals, cv=5, scoring='r2')

ridge.fit(X_scaled, y_vals)
lasso.fit(X_scaled, y_vals)

print('=== Regularized Regression (5-Fold CV R²) ===')
print(f'Ridge  : mean={ridge_cv.mean():.4f}  std={ridge_cv.std():.4f}')
print(f'Lasso  : mean={lasso_cv.mean():.4f}  std={lasso_cv.std():.4f}')

# Lasso zeroed features
lasso_coef = pd.DataFrame({'feature': reg_features, 'coef': lasso.coef_})
zeroed     = lasso_coef[lasso_coef['coef'] == 0]['feature'].tolist()
active     = lasso_coef[lasso_coef['coef'] != 0]['feature'].tolist()
print(f'\nLasso zeroed out  : {zeroed}')
print(f'Lasso kept active : {active}')

In [ ]:
# Cell 12 — Residual Diagnostics
fitted    = ols_model.fittedvalues
residuals = ols_model.resid

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.3, color='#1a936f', s=10)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Residuals vs Fitted', fontweight='bold')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')

# Residual Distribution
axes[1].hist(residuals, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

# QQ Plot
qqplot(residuals, line='s', ax=axes[2], alpha=0.4)
axes[2].set_title('Q-Q Plot (Normality Check)', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

# Breusch-Pagan test for heteroscedasticity
X_const = sm.add_constant(df[reg_features])
bp_test = het_breuschpagan(residuals, X_const)
print('=== Breusch-Pagan Heteroscedasticity Test ===')
print(f'LM Statistic : {bp_test[0]:.4f}')
print(f'P-value      : {bp_test[1]:.4f}')
print('→', 'Heteroscedasticity DETECTED' if bp_test[1] < 0.05 else 'No significant heteroscedasticity')

In [ ]:
# Cell 13 — Promotional Lift Analysis (Holiday Season)
holiday_sales     = df[df['is_holiday_season'] == 1]['sales']
non_holiday_sales = df[df['is_holiday_season'] == 0]['sales']

t_stat, t_pval = stats.ttest_ind(holiday_sales, non_holiday_sales)
lift = (holiday_sales.mean() - non_holiday_sales.mean()) / non_holiday_sales.mean() * 100

print('=== Holiday Season Promotional Lift ===')
print(f'Holiday avg sales     : {holiday_sales.mean():.2f}')
print(f'Non-holiday avg sales : {non_holiday_sales.mean():.2f}')
print(f'Promotional Lift      : +{lift:.1f}%')
print(f'T-statistic           : {t_stat:.4f}')
print(f'P-value               : {t_pval:.4e}')
print('→ Lift is', 'STATISTICALLY SIGNIFICANT' if t_pval < 0.05 else 'not significant')

fig = px.box(df, x='is_holiday_season', y='sales',
             title=f'Holiday vs Non-Holiday Sales — Lift: +{lift:.1f}%',
             labels={'is_holiday_season': 'Holiday Season (1=Yes)', 'sales': 'Sales'},
             color='is_holiday_season',
             color_discrete_map={0: '#3498db', 1: '#e74c3c'})
fig.update_layout(height=400, plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()

In [ ]:
# Cell 14 — Save Regression Results
import os
os.makedirs('../data/processed', exist_ok=True)

# Save significant features for supply chain notebook
sig_features = sig.index.tolist()
reg_summary  = pd.DataFrame({
    'feature':     ols_model.params.index[1:],
    'coefficient': ols_model.params.values[1:],
    'p_value':     ols_model.pvalues.values[1:],
    'significant': ols_model.pvalues.values[1:] < 0.05
})
reg_summary.to_csv('../data/processed/regression_summary.csv', index=False)

print(' regression_summary.csv saved')
print()
print('=== Regression Summary ===')
print(f'R-squared             : {ols_model.rsquared:.4f}')
print(f'Adj R²                : {ols_model.rsquared_adj:.4f}')
print(f'Significant features  : {len(sig)} / {len(reg_features)}')
print(f'Price elasticity      : {slope:.3f}')
print(f'Holiday lift          : +{lift:.1f}%')
print(f'Ridge CV R²           : {ridge_cv.mean():.4f}')
print(f'Lasso CV R²           : {lasso_cv.mean():.4f}')